# MachineLearningCVE: feature selection → synthetic data → similarity

Apply the IoT-23 pipeline to the eight traffic CSVs in `data/MachineLearningCVE`.

**Pipeline:** load → sample → clean → 70/30 stratified split → five TRAIN-only rankings (ANOVA, chi-square/Cramér's V, MI, RFE, RF importance) → vote → CTGAN on all versus selected features → held-out similarity and TSTR macro-F1.

The original attack labels are retained (multiclass). Both generators are evaluated on the same selected features plus label. Outputs go to `results_machinelearning/`.

Run with the project's `.venv` Python kernel (SDV, SDMetrics, PyTorch, pandas, NumPy, SciPy, scikit-learn, matplotlib). **Run All includes lengthy CTGAN training.** For a smoke run set `EPOCHS=1`, `MAX_ROWS_PER_CLASS=100`, and `RFE_STEP=0.2`; these are debugging settings, not research results.


## 0. Setup and settings

In [1]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE
from sklearn.preprocessing import KBinsDiscretizer
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from sdmetrics.single_column import KSComplement, TVComplement
from sdmetrics.column_pairs import CorrelationSimilarity, ContingencySimilarity

from pathlib import Path

PROJECT = Path.cwd()
if not (PROJECT / "data" / "MachineLearningCVE").exists():
    raise FileNotFoundError("Start this notebook from the AdaptSynth project directory.")
DATA_DIR = PROJECT / "data" / "MachineLearningCVE"
OUTPUT_DIR = PROJECT / "results_machinelearning"
N_SELECT = 10
EPOCHS = 200
SEEDS = [42]                  # use [42, 43, 44] for repeated generator runs
SPLIT_SEED = 42
MAX_ROWS_PER_CLASS = 2000      # None reads ALL rows; can require substantial RAM/time
CHUNK_SIZE = 100_000
RFE_STEP = 1                  # same elimination rule as IoT-23; 0.1 is faster


ModuleNotFoundError: No module named 'torch'

## 1. Load the traffic CSVs

Headers have leading spaces and labels may contain non-UTF-8 characters; normalize these on read. Scan every file in chunks. Keep up to `MAX_ROWS_PER_CLASS` rows per original label using random priorities (a uniform sample within each class). This retains rare attacks but **changes class proportions**; scores describe this sampled population, not the original traffic prevalence. Set the cap to `None` to use the complete dataset.

`source_counts` records original label counts and `sample_counts` records the sample. The same sample and split are used for both generators. Source filenames are never used as predictors.


In [ ]:
files = sorted(DATA_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSVs found in {DATA_DIR}")
rng = np.random.default_rng(SPLIT_SEED)
reservoirs, source_counts, all_chunks = {}, {}, []
for path in files:
    for chunk in pd.read_csv(path, encoding="cp1252", chunksize=CHUNK_SIZE, low_memory=False):
        chunk.columns = chunk.columns.str.strip()
        if chunk.columns.duplicated().any():
            raise ValueError(f"Duplicate headers after normalization: {path.name}")
        chunk = chunk.rename(columns={"Label": "label"})
        if "label" not in chunk:
            raise ValueError(f"Missing Label in {path.name}")
        chunk = chunk.dropna(subset=["label"])
        chunk["label"] = chunk["label"].astype(str).str.strip()
        chunk = chunk.loc[chunk["label"].ne("")].copy()
        for label, group in chunk.groupby("label", sort=False):
            source_counts[label] = source_counts.get(label, 0) + len(group)
            if MAX_ROWS_PER_CLASS is None:
                all_chunks.append(group)
            else:
                group = group.copy()
                group["_priority"] = rng.random(len(group))
                previous = reservoirs.get(label)
                pool = pd.concat([previous, group], ignore_index=True) if previous is not None else group
                reservoirs[label] = pool.nsmallest(MAX_ROWS_PER_CLASS, "_priority")
    print("Read", path.name, flush=True)
raw = (pd.concat(all_chunks, ignore_index=True) if MAX_ROWS_PER_CLASS is None else
       pd.concat(reservoirs.values(), ignore_index=True).drop(columns="_priority"))
sample_counts = raw["label"].value_counts()
print("Sample shape:", raw.shape)
display(pd.DataFrame({"source_rows": pd.Series(source_counts), "sample_rows": sample_counts}))
raw.head()


## 2. Normalize predictors and remove exact duplicate rows

Use `label` as the multiclass target and numeric traffic measurements as predictors. Convert invalid numeric values and infinities to missing values. Remove identical predictor/label rows before splitting so the same recorded row cannot occur in train and test. Sampling occurs before deduplication, so this is not a uniform sample of unique flows.

Training-only preprocessing in the next step removes empty/constant features and duplicate feature columns, and learns median imputation. No test values influence those decisions. A random flow split does not establish generalization to unseen days or devices.


In [ ]:
df = raw.copy()
identifier_columns = [c for c in ["Flow ID", "Source IP", "Destination IP", "Timestamp"] if c in df]
df = df.drop(columns=identifier_columns)
for c in df.columns.drop("label"):
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan)
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Exact duplicate rows removed:", before - len(df))
counts = df["label"].value_counts()
if len(counts) < 2 or counts.min() < 4:
    raise ValueError("Need at least two classes and four distinct rows per class; increase the sample cap.")
print(counts)


## 3. Split and fit preprocessing on TRAIN only

Stratify by original attack label. Fit column removal and median imputation using train only; apply the same transformation to test. Limit numeric magnitude to float32-safe values for scikit-learn. Destination Port is treated as categorical; all other predictors are numeric.


In [ ]:
train, test = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SPLIT_SEED)
train, test = train.reset_index(drop=True), test.reset_index(drop=True)
candidates = train.columns.drop("label").tolist()
# Values outside float32 range cannot be used by the ranking/classification forests.
limit = np.finfo(np.float32).max / 100
train[candidates] = train[candidates].clip(-limit, limit)
test[candidates] = test[candidates].clip(-limit, limit)
constant = [c for c in candidates if train[c].nunique(dropna=True) <= 1]
train, test = train.drop(columns=constant), test.drop(columns=constant)
# Some CSV versions contain a repeated Fwd Header Length column.
features = train.columns.drop("label").tolist()
duplicate_columns = train[features].T.duplicated()
duplicates = duplicate_columns.index[duplicate_columns].tolist()
train, test = train.drop(columns=duplicates), test.drop(columns=duplicates)
FEATURES = train.columns.drop("label").tolist()
medians = train[FEATURES].median()
train[FEATURES] = train[FEATURES].fillna(medians)
test[FEATURES] = test[FEATURES].fillna(medians)
CATEGORICAL = [c for c in ["Destination Port", "label"] if c in train]
NUMERICAL = [c for c in FEATURES if c not in CATEGORICAL]
for c in CATEGORICAL:
    train[c], test[c] = train[c].astype(str), test[c].astype(str)
if not 1 < N_SELECT < len(FEATURES):
    raise ValueError(f"N_SELECT must be between 2 and {len(FEATURES)-1}")
assert np.isfinite(train[NUMERICAL].to_numpy()).all()
assert np.isfinite(test[NUMERICAL].to_numpy()).all()
print("Dropped constant/empty:", constant)
print("Dropped duplicate columns:", duplicates)
print("Train", train.shape, "Test", test.shape, "Candidate features", len(FEATURES))


## 4. Rank features with five techniques (TRAIN only)

Feature-selection methods come in three families. We use all three, so the vote is not biased toward one way of thinking.

| Family | How it works | Technique used here |
|---|---|---|
| **Filter** | Scores each feature with a statistic, **without training any model**. Fast, but ignores feature interactions. | **ANOVA F-test**, **Chi-square test**, **Mutual information** |
| **Wrapper** | Repeatedly **trains a model** on different feature subsets and keeps the subset that works best. Slower, accounts for interactions. | **RFE** (Recursive Feature Elimination) with a random forest |
| **Embedded** | Selection happens **inside model training**; the trained model reports which features it relied on. | **Random-forest importance** (how much each feature reduces impurity across all trees) |

### How the filter statistics work

* **ANOVA F-test** — for one numeric feature, compare *how far apart the class averages are* with *how spread out values are inside each class*:
  `F = (variance BETWEEN class means) / (variance WITHIN classes)`.
  Big F = attack-class flows have clearly different averages for this feature. It only sees differences in the **mean** (a linear/average effect), assumes roughly bell-shaped data, and needs **numeric** features.
* **Chi-square test (χ²)** — for one feature vs the label, build a table of counts (feature value × attack label) and ask: *are these counts different from what we would see if the feature and label were unrelated?*
  `χ² = Σ (observed − expected)² / expected`. Needs **categories**, so numeric features are first cut into 10 equal-size bins. The raw χ² grows with the number of categories, so we convert it to **Cramér's V** (0 = unrelated, 1 = perfectly related), which is comparable across features.
* **Mutual information (MI)** — measures how much knowing the feature **reduces uncertainty about the label**:
  `MI = Σ p(x,y) · log[ p(x,y) / (p(x)·p(y)) ]`.
  It is 0 when the feature and label are independent and grows as they become more dependent. Unlike ANOVA it also catches **non-linear** relationships (e.g. "malicious only when the port is exactly 23").

The models need numbers, so categorical features are turned into integer codes (`OrdinalEncoder`) for ANOVA / MI / RFE / forest. Chi-square uses the categories directly.


In [ ]:
X = train[FEATURES].copy()
y = train["label"]

cat_features = [c for c in FEATURES if c in CATEGORICAL]
num_features = [c for c in FEATURES if c in NUMERICAL]
X[cat_features] = OrdinalEncoder().fit_transform(X[cat_features])      # integer codes (ranking only)
is_discrete = [c in CATEGORICAL for c in FEATURES]

# ---------------- FILTER 1: ANOVA F-test (numeric features only) ----------------
print("Ranking 1/5: ANOVA", flush=True)
anova = pd.Series(f_classif(X[num_features], y)[0], index=num_features).reindex(FEATURES)   # NaN for categoricals

# ---------------- FILTER 2: Chi-square -> Cramer's V (all features) ----------------
# numeric features are binned into 10 quantile bins so they become categories
print("Ranking 2/5: chi-square", flush=True)
binned = X.copy()
binned[num_features] = KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="quantile",
                                        subsample=None).fit_transform(X[num_features])
def cramers_v(feature):
    table = pd.crosstab(binned[feature], y)                 # counts: feature value x label
    if min(table.shape) < 2:                                # feature fell into a single bin (almost always 0): no information
        return 0.0
    chi2 = chi2_contingency(table, correction=False)[0]
    return np.sqrt(chi2 / (len(y) * (min(table.shape) - 1)))  # scaled to 0..1
chi = pd.Series({f: cramers_v(f) for f in FEATURES})

# ---------------- FILTER 3: Mutual information (all features) ----------------
print("Ranking 3/5: mutual information", flush=True)
mi = pd.Series(mutual_info_classif(X, y, discrete_features=is_discrete, random_state=42), index=FEATURES)

# ---------------- WRAPPER: RFE with a random forest ----------------
# train a forest, drop the weakest feature, retrain, repeat until N_SELECT features remain.
# ranking_ = 1 for the survivors, 2 for the last one dropped, 3 for the one before, ...
print("Ranking 4/5: RFE (repeated forest fits; this can be slow)", flush=True)
rfe = RFE(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
          n_features_to_select=N_SELECT, step=RFE_STEP).fit(X, y)
rfe_score = pd.Series(-rfe.ranking_.astype(float), index=FEATURES)   # negate so higher = better

# ---------------- EMBEDDED: random-forest impurity importance ----------------
print("Ranking 5/5: forest importance", flush=True)
forest = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X, y)
embedded = pd.Series(forest.feature_importances_, index=FEATURES)

scores = pd.DataFrame({"anova_f (filter)": anova, "chi2_cramersV (filter)": chi, "mutual_info (filter)": mi,
                       "rfe (wrapper)": rfe_score, "rf_importance (embedded)": embedded})
scores.sort_values("mutual_info (filter)", ascending=False).round(4)


## 5. Voting

Each technique looks at its own ranking and gives **one vote** to each of its top `N_SELECT` features
(so a feature can get at most 5 votes). Features are sorted by votes; ties are broken by **average rank** (lower = better).
This is a rule we are *testing*, not a guarantee that the chosen subset is optimal.

In [ ]:
# 1 = best; NaN stays NaN (ANOVA has no score for categoricals).
# method="first" ranks tied scores one after another so each technique hands out exactly N_SELECT votes.
ranks = scores.rank(ascending=False, method="first")
# RFE's survivors are genuinely tied at rank 1, so give them all rank 1 (still exactly N_SELECT of them)
ranks["rfe (wrapper)"] = scores["rfe (wrapper)"].rank(ascending=False, method="min")
votes = (ranks <= N_SELECT).astype(int)

vote_table = votes.add_prefix("vote_").join(ranks.add_prefix("rank_"))
vote_table["votes"] = votes.sum(axis=1)
vote_table["avg_rank"] = ranks.mean(axis=1)                 # mean over the techniques that could score the feature
vote_table = vote_table.sort_values(["votes", "avg_rank"], ascending=[False, True])
vote_table["selected"] = [i < N_SELECT for i in range(len(vote_table))]

SELECTED = vote_table.index[:N_SELECT].tolist()
print("Selected features:", SELECTED)
vote_table[["votes", "avg_rank", "selected"] + [c for c in vote_table.columns if c.startswith("vote_")]]
# Checkpoint feature selection before the lengthy generator training.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
scores.to_csv(OUTPUT_DIR / "feature_scores.csv")
vote_table.to_csv(OUTPUT_DIR / "voting_table.csv")
pd.Series(SELECTED, name="feature").to_csv(OUTPUT_DIR / "selected_features.csv", index=False)


In [ ]:
vote_table["votes"].iloc[::-1].plot.barh(
    color=["#2a7" if s else "#bbb" for s in vote_table["selected"].iloc[::-1]],
    figsize=(9, max(5, len(FEATURES) * 0.18)), title="Votes per feature (green = selected)")
plt.xlabel("votes (max 5)"); plt.tight_layout(); plt.show()


## 6. Train CTGAN twice

* **Baseline**: all candidate features + `label`.
* **Selected**: only the voted features + `label`.

Same generator, same settings, same seed, so the *only* difference is the column set.
Each synthetic table has as many rows as the training table.

In [ ]:
def train_and_sample(cols, seed):
    """Fit single-table CTGAN; return synthetic rows and fit time (sampling excluded)."""
    import random
    data = train[cols].copy()
    meta = Metadata.detect_from_dataframes({"flows": data})
    for c in cols:
        meta.update_column(table_name="flows", column_name=c, sdtype="categorical" if c in CATEGORICAL else "numerical")
    random.seed(seed); torch.manual_seed(seed); np.random.seed(seed)
    synth = CTGANSynthesizer(meta, epochs=EPOCHS, batch_size=500, verbose=True)
    # Global Python/NumPy/PyTorch seeds above control fitting; SDV 2 has no public set_random_state.
    t0 = time.perf_counter()
    synth.fit({"flows": data})
    seconds = time.perf_counter() - t0
    return synth.sample(num_rows=len(data))["flows"], seconds

experiments = {"baseline (all features)": FEATURES + ["label"],
               "selected (voted top-N)": SELECTED + ["label"]}
synthetic, runtime = {}, {}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for seed in SEEDS:
    for name, cols in experiments.items():
        print(f"Starting {name}, seed={seed}, epochs={EPOCHS}, rows={len(train)}, columns={len(cols)}", flush=True)
        synthetic[(name, seed)], runtime[(name, seed)] = train_and_sample(cols, seed)
        synthetic[(name, seed)].to_csv(OUTPUT_DIR / f"synthetic_{name.split()[0]}_seed{seed}.csv", index=False)
        print(f"{name:26s} seed={seed} columns={len(cols):2d} {runtime[(name, seed)]:.0f}s")


## 7. Compare synthetic vs real (against the held-out TEST table)

We never compare row 1 with row 1 — synthetic rows have no matching real row. We compare **distributions** and **relationships**.

**Method A — column distributions** (each column on its own), score 0–1, higher = more similar
* numeric columns → `KSComplement` = 1 − KS statistic (largest gap between the two cumulative distributions)
* categorical columns → `TVComplement` = 1 − total variation distance (gap between category proportions)

**Method B — relationships between columns**, score 0–1, higher = more similar
* numeric pairs → `CorrelationSimilarity` = 1 − |corr_real − corr_synth| / 2
* categorical pairs → `ContingencySimilarity` (compares the 2-way count tables)

**Method C (bonus) — TSTR**: train a classifier on the synthetic table, test it on real data (the protocol from your paper). It measures usefulness, not just resemblance.

To compare the two experiments **fairly, both are scored on the same columns** (the selected ones + label).
A score of 0.90 is a similarity score, not "90% accuracy". None of this measures privacy.

In [ ]:
def similarity(real, syn, cols):
    """Return average column-distribution and relationship similarity over `cols`."""
    num = [c for c in cols if c in NUMERICAL]
    cat = [c for c in cols if c in CATEGORICAL]

    # Method A: per-column scores
    per_col = {c: KSComplement.compute(real[c], syn[c]) for c in num}
    per_col.update({c: TVComplement.compute(real[c], syn[c]) for c in cat})

    # Method B: per-pair scores (pairs that are constant in either table give NaN and are skipped)
    pairs = []
    for i, a in enumerate(num):
        for b in num[i + 1:]:
            pairs.append(CorrelationSimilarity.compute(real[[a, b]], syn[[a, b]]))
    for i, a in enumerate(cat):
        for b in cat[i + 1:]:
            pairs.append(ContingencySimilarity.compute(real[[a, b]], syn[[a, b]]))

    return {"numerical_cols": np.mean([per_col[c] for c in num]),
            "categorical_cols": np.mean([per_col[c] for c in cat]),
            "relationships": np.nanmean(pairs)}, per_col

def tstr(syn, cols):
    """Train on synthetic, test on real test set; macro-F1."""
    feats = [c for c in cols if c != "label"]
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cf = [c for c in feats if c in CATEGORICAL]
    Xs, Xt = syn[feats].copy(), test[feats].copy()
    if cf:
        Xs[cf] = enc.fit_transform(Xs[cf]); Xt[cf] = enc.transform(Xt[cf])
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xs, syn["label"])
    return f1_score(test["label"], clf.predict(Xt), average="macro", labels=sorted(train["label"].unique()), zero_division=0)

COMMON = SELECTED + ["label"]     # same columns for both experiments -> fair comparison
rows, per_column = [], {}
for (name, seed), syn in synthetic.items():
    s, pc = similarity(test, syn, COMMON)
    per_column[(name, seed)] = pc
    rows.append({"experiment": name, "seed": seed, "columns_trained_on": syn.shape[1],
                 **s, "TSTR_macroF1": tstr(syn, COMMON), "train_seconds": runtime[(name, seed)]})
results = pd.DataFrame(rows)
results


### Results table (mean over seeds; ± std appears only when you use more than one seed)

In [ ]:
summary = results.drop(columns="seed").groupby("experiment").agg(["mean", "std"]).round(3)
summary

In [ ]:
ref, _ = similarity(test, train, COMMON)
print("Real train vs real test similarity:", {k: round(v, 3) for k, v in ref.items()})
real_train_f1 = tstr(train[COMMON], COMMON)
print("Real-trained classifier reference macro-F1:", round(real_train_f1, 3))
# This is a reference, not a mathematical upper bound on TSTR.


## 8. Look at it, not just the numbers

Left: real test correlations; middle/right: synthetic correlations for each experiment (numeric selected columns).
Similar colors = relationships were preserved.

In [1]:
num_common = [c for c in COMMON if c in NUMERICAL]
seed0 = SEEDS[0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (title, frame) in zip(axes, [("REAL (test)", test)] +
                                     [(n, synthetic[(n, seed0)]) for n in experiments]):
    corr = frame[num_common].corr()
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(num_common))); ax.set_xticklabels(num_common, rotation=90)
    ax.set_yticks(range(len(num_common))); ax.set_yticklabels(num_common)
    ax.set_title(title)
fig.colorbar(im, ax=axes, shrink=0.8); plt.show()

NameError: name 'COMMON' is not defined

In [ ]:
# per-column scores: which individual columns did each experiment reproduce well?
pd.DataFrame({name: per_column[(name, seed0)] for name in experiments}).round(3)

## 9. Save outputs

Save rankings, votes, selected features, preprocessing parameters, source/sample counts, per-column scores, per-seed comparison, summary, references, and run configuration to `results_machinelearning/`. Synthetic tables are checkpointed immediately after each generator run.


In [ ]:
import json
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
scores.to_csv(OUTPUT_DIR / "feature_scores.csv")
vote_table.to_csv(OUTPUT_DIR / "voting_table.csv")
pd.Series(SELECTED, name="feature").to_csv(OUTPUT_DIR / "selected_features.csv", index=False)
results.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv")
pd.DataFrame(per_column).to_csv(OUTPUT_DIR / "per_column_similarity.csv")
pd.DataFrame({"source_rows": pd.Series(source_counts), "sample_rows": sample_counts,
              "clean_rows": df["label"].value_counts(), "train_rows": train["label"].value_counts(),
              "test_rows": test["label"].value_counts()}).to_csv(OUTPUT_DIR / "class_counts.csv")
medians.to_csv(OUTPUT_DIR / "training_medians.csv")
config = {"dataset": str(DATA_DIR), "files": [p.name for p in files],
          "max_rows_per_class": MAX_ROWS_PER_CLASS, "epochs": EPOCHS, "seeds": SEEDS,
          "split_seed": SPLIT_SEED, "n_select": N_SELECT, "rfe_step": RFE_STEP,
          "selected_features": SELECTED, "dropped_constants": constant,
          "dropped_duplicate_columns": duplicates, "real_similarity": ref,
          "real_train_macroF1": real_train_f1,
          "relationship_scope": "numeric-numeric and categorical-categorical only"}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2))
print("Saved to", OUTPUT_DIR)


## Interpretation and limits

* Similar fidelity and TSTR with less fit time supports feature selection for this sampled experiment. Report per-seed results and mean ± standard deviation; one seed is preliminary.
* The per-class cap changes prevalence. Macro-F1 weights classes equally; it does not estimate operational accuracy under the original class imbalance.
* RFE and feature selection use a fixed training split. Multiple generator seeds measure generator variability, not split or feature-selection stability.
* The random flow split can share capture conditions between train and test. A separate day/capture split is needed for claims about unseen environments.
* Relationship similarity here averages numeric-numeric and categorical-categorical pairs. It omits mixed numeric/categorical pairs and does not fully measure feature/label dependence.
* Real-train/test similarity and real-trained classifier F1 are reference points, not guaranteed upper bounds. Fidelity does not measure privacy.
* Compare settings and sampling policies when presenting these results beside the IoT-23 experiment. Training time excludes sampling, ranking, preprocessing, and scoring.


In [ ]:
# Log this run to paper_results/experiment_log.csv and rebuild the paper workbook.
# config_tag: set a short label (e.g. "full data") when you change settings like MAX_ROWS_PER_CLASS,
# so those runs show up as separate rows instead of replacing earlier ones.
import sys
sys.path.insert(0, str(PROJECT / "src"))
from results_log import log_comparison
from build_results_workbook import build
log_comparison(results, dataset="CIC-IDS2017", generator="CTGAN", classifier="RandomForest",
               epochs=EPOCHS, trtr_f1=real_train_f1, config_tag="", source=OUTPUT_DIR.name)
build()